In [2]:
import pandas as pd
import numpy as np
import re
import string
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, PassiveAggressiveClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [3]:
# 1. Load Datasets
df_fake = pd.read_csv('Fake.csv')
df_true = pd.read_csv('True.csv')

In [5]:
df_fake

,title,text,subject,date
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017"
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017"
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017"
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017"
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017"
...,...,...,...,...
23476,McPain: John McCain Furious That Iran Treated ...,21st Century Wire says As 21WIRE reported earl...,Middle-east,"January 16, 2016"
23477,JUSTICE? Yahoo Settles E-mail Privacy Class-ac...,21st Century Wire says It s a familiar theme. ...,Middle-east,"January 16, 2016"
23478,Sunnistan: US and Allied ‘Safe Zone’ Plan to T...,Patrick Henningsen 21st Century WireRemember ...,Middle-east,"January 15, 2016"
23479,How to Blow $700 Million: Al Jazeera America F...,21st Century Wire says Al Jazeera America will...,Middle-east,"January 14, 2016"


In [6]:
df_true

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"
...,...,...,...,...
21412,'Fully committed' NATO backs new U.S. approach...,BRUSSELS (Reuters) - NATO allies on Tuesday we...,worldnews,"August 22, 2017"
21413,LexisNexis withdrew two products from Chinese ...,"LONDON (Reuters) - LexisNexis, a provider of l...",worldnews,"August 22, 2017"
21414,Minsk cultural hub becomes haven from authorities,MINSK (Reuters) - In the shadow of disused Sov...,worldnews,"August 22, 2017"
21415,Vatican upbeat on possibility of Pope Francis ...,MOSCOW (Reuters) - Vatican Secretary of State ...,worldnews,"August 22, 2017"


In [7]:
# 2. Add Target Labels (0 = Fake, 1 = Real)
df_fake['label'] = 0
df_true['label'] = 1

In [8]:
df_fake.columns.tolist()

['title', 'text', 'subject', 'date', 'label']

In [9]:
df_true.columns.tolist()

['title', 'text', 'subject', 'date', 'label']

In [10]:
# Combine title and text into a single column for richer context
df_fake['content'] = df_fake['title'].fillna('') + ' ' + df_fake['text'].fillna('')
df_true['content'] = df_true['title'].fillna('') + ' ' + df_true['text'].fillna('')

In [11]:
# Merge datasets and shuffle rows
df = pd.concat([df_fake[['content', 'label']], df_true[['content', 'label']]], axis=0)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

In [12]:
df

,content,label
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0
1,Trump drops Steve Bannon from National Securit...,1
2,Puerto Rico expects U.S. to lift Jones Act shi...,1
3,OOPS: Trump Just Accidentally Confirmed He Le...,0
4,Donald Trump heads for Scotland to reopen a go...,1
...,...,...
44893,UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...,0
44894,PM May seeks to ease Japan's Brexit fears duri...,1
44895,Merkel: Difficult German coalition talks can r...,1
44896,Trump Stole An Idea From North Korean Propaga...,0


In [13]:
# 3. Text Preprocessing Function
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)  # Remove URLs
    text = re.sub(r'<.*?>+', '', text)                 # Remove HTML tags
    text = re.sub(r'[%s]' % re.escape(string.punctuation), '', text) # Remove punctuation
    text = re.sub(r'\n', ' ', text)                    # Remove newlines
    text = re.sub(r'\w*\d\w*', '', text)               # Remove words containing numbers
    text = re.sub(r'\s+', ' ', text).strip()           # Remove extra spaces
    return text

df['cleaned_content'] = df['content'].apply(clean_text)

In [15]:
# 4. Train / Test Split
X = df['cleaned_content']
y = df['label']

In [17]:
X

0        ben stein calls out circuit court committed a ...
1        trump drops steve bannon from national securit...
2        puerto rico expects us to lift jones act shipp...
3        oops trump just accidentally confirmed he leak...
4        donald trump heads for scotland to reopen a go...
                               ...                        
44893    unreal cbs’s ted koppel tells sean hannity he’...
44894    pm may seeks to ease japans brexit fears durin...
44895    merkel difficult german coalition talks can re...
44896    trump stole an idea from north korean propagan...
44897    breaking hillary clinton’s state department ga...
Name: cleaned_content, Length: 44898, dtype: object

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)

In [19]:
# 5. TF-IDF Feature Extraction
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [20]:
# 6. Train Multiple Models in a Loop
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Passive Aggressive Classifier': PassiveAggressiveClassifier(max_iter=1000, random_state=42),
    'Multinomial Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
}

In [21]:
best_model = None
best_accuracy = 0.0
best_model_name = ""

In [22]:
print("--- Model Performance Comparison ---")
for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    predictions = model.predict(X_test_tfidf)
    acc = accuracy_score(y_test, predictions)
    print(f"{name} Accuracy: {acc * 100:.2f}%")
    
    if acc > best_accuracy:
        best_accuracy = acc
        best_model = model
        best_model_name = name

print(f"\nBest Model: {best_model_name} with Accuracy: {best_accuracy * 100:.2f}%")

--- Model Performance Comparison ---
Logistic Regression Accuracy: 98.94%
Passive Aggressive Classifier Accuracy: 99.50%
Multinomial Naive Bayes Accuracy: 94.97%
Random Forest Accuracy: 99.71%

Best Model: Random Forest with Accuracy: 99.71%


In [23]:
# Detailed Evaluation of Best Model
best_preds = best_model.predict(X_test_tfidf)
print("\nClassification Report:")
print(classification_report(y_test, best_preds, target_names=['Fake', 'Real']))


Classification Report:
              precision    recall  f1-score   support

        Fake       1.00      1.00      1.00      4696
        Real       1.00      1.00      1.00      4284

    accuracy                           1.00      8980
   macro avg       1.00      1.00      1.00      8980
weighted avg       1.00      1.00      1.00      8980



In [24]:
# 7. Save Model & Vectorizer
joblib.dump(best_model, 'best_model.pkl')
joblib.dump(vectorizer, 'tfidf_vectorizer.pkl')
print("Model and Vectorizer successfully saved to disk.")

Model and Vectorizer successfully saved to disk.


In [25]:
df

,content,label,cleaned_content
0,Ben Stein Calls Out 9th Circuit Court: Committ...,0,ben stein calls out circuit court committed a ...
1,Trump drops Steve Bannon from National Securit...,1,trump drops steve bannon from national securit...
2,Puerto Rico expects U.S. to lift Jones Act shi...,1,puerto rico expects us to lift jones act shipp...
3,OOPS: Trump Just Accidentally Confirmed He Le...,0,oops trump just accidentally confirmed he leak...
4,Donald Trump heads for Scotland to reopen a go...,1,donald trump heads for scotland to reopen a go...
...,...,...,...
44893,UNREAL! CBS’S TED KOPPEL Tells Sean Hannity He...,0,unreal cbs’s ted koppel tells sean hannity he’...
44894,PM May seeks to ease Japan's Brexit fears duri...,1,pm may seeks to ease japans brexit fears durin...
44895,Merkel: Difficult German coalition talks can r...,1,merkel difficult german coalition talks can re...
44896,Trump Stole An Idea From North Korean Propaga...,0,trump stole an idea from north korean propagan...
